In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=7737900c533ccf84ec0c3ba6fc988efe2c8b5a5d65c1d77f03bbb7f431c7ee62
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [32]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import numpy as np

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

def random():
  circuit = QuantumCircuit(1,1)
  circuit.h(0)

  state = Statevector.from_int(0,2)
  state = state.evolve(circuit)
  counts = state.sample_counts(1)
  return int(list(counts.keys())[0])

def entangled_pair():
  circuit = QuantumCircuit(2)
  circuit.h(0)
  circuit.cx(0,1)
  return circuit

def random_3_choice():
  state = Statevector([1/math.sqrt(3), math.sqrt(2)/math.sqrt(3)])
  counts = state.sample_counts(1)

  qubit = int(list(counts.keys())[0])

  if qubit == 0:
    return 0

  state = Statevector([1/math.sqrt(2), 1/math.sqrt(2)])
  counts = state.sample_counts(1)
  qubit = int(list(counts.keys())[0])
  if qubit == 0:
    return 1
  else:
    return 2

def apply_measurement(circuit, qubit, basis):

    if basis == 'X':
        circuit.h(qubit)

    elif basis == 'W':
        circuit.ry(-math.pi/4, qubit)

    elif basis == 'V':
        circuit.ry(math.pi/4, qubit)

    elif basis == 'Z':
        pass

def match(results, alice_choice, bob_choice):
  matched = [result['alice_bit'] * result['bob_bit']
             for result in results
             if result['alice_choice'] == alice_choice and result['bob_choice'] == bob_choice
            ]
  if len(matched) == 0:
    return 0

  return sum(matched) / len(matched)

def calculate_S(qubits):
  xw = match(qubits, 'X','W')
  xv = match(qubits, 'X','V')
  zw = match(qubits, 'Z','W')
  zv = match(qubits, 'Z','V')

  S = abs(xw - xv + zw + zv)
  return S

N = 200

a_ops = ['X','W','Z']
b_ops = ['W', 'Z', 'V']
sequence = []

for i in range(int(9 * N /2)):
  pair = entangled_pair()
  state = Statevector.from_int(0,4)
  state = state.evolve(pair)

  alice_i = random_3_choice()
  bob_j = random_3_choice()

  a_op = a_ops[alice_i]
  b_op = b_ops[bob_j]

  circuit = QuantumCircuit(2)
  apply_measurement(circuit, [1], a_op)
  apply_measurement(circuit, [0], b_op)

  state = state.evolve(circuit)
  counts = state.sample_counts(1)
  result = list(counts.keys())[0]

  alice_bit = (-1) ** int(result[0])
  bob_bit = (-1) ** int(result[1])
  sequence.append({'alice_i' : alice_i, 'alice_choice': a_ops[alice_i], 'alice_bit': alice_bit,
                   'bob_j': bob_j, 'bob_choice': b_ops[bob_j], 'bob_bit': bob_bit})


final_key = []
S_list = []
for i in sequence:
  if i['alice_i'] == 1 and i['bob_j'] == 0 or i['alice_i'] == 2 and i['bob_j'] == 1:
    final_key.append(i['alice_bit'])
  else:
    S_list.append(i)

print(f"Final key : {final_key}")
S = calculate_S(S_list)
print(f"S = {S:.3f}")
if abs(S - 2 * math.sqrt(2)) < 0.5:
  print("Entanlged confirmed")
  print(f"Shared key: {final_key}")
elif abs(S) <= 2:
  print(f"S  = {S:.3f} <= 2, complete quantum correlation lost")
else:
  print(f"S = {S:.3f}, significant difference to 2 root 2, possible eavesdropper")

Final key : [-1, -1, -1, -1, -1, 1, -1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, -1, -1, 1, -1, 1, -1, 1, 1, -1, 1, -1, -1, 1, -1, 1, -1, -1, 1, -1, -1, 1, 1, 1, 1, -1, -1, 1, 1, -1, -1, -1, -1, 1, 1, -1, -1, 1, -1, 1, 1, -1, -1, -1, -1, -1, 1, 1, 1, -1, -1, 1, 1, -1, -1, 1, -1, -1, -1, 1, 1, 1, -1, -1, -1, 1, -1, -1, 1, 1, 1, 1, 1, -1, -1, 1, -1, 1, -1, -1, 1, 1, 1, 1, -1, 1, -1, -1, -1, 1, -1, -1, -1, -1, 1, 1, -1, 1, 1, -1, -1, 1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -1, 1, -1, -1, -1, 1, -1, -1, -1, 1, -1, 1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, 1, -1, -1, -1, -1, 1, 1, 1, -1, -1, 1, -1, 1, 1, -1, 1, -1, -1, -1, -1, -1, -1, -1, 1, 1, -1, 1, -1, 1, -1, 1, 1, 1, 1, -1, 1, -1, -1, 1, -1, 1, 1, 1, 1, 1, -1, -1, 1, -1, 1, -1, 1, -1, -1, -1, -1]
S = 2.954
Entanlged confirmed
Shared key: [-1, -1, -1, -1, -1, 1, -1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, -1, -1, 1, -1, 1, -1, 1, 1, -1, 1, -1, -1, 1, -1, 1, -1, -1, 1, -1, -1, 1, 1, 1, 1, -1, -1, 1, 1, -1, -1, -1, -1, 1, 1, -1, -1, 1, -1, 1, 1, -1, -1, 